In [ ]:
import zipfile
import os

def extract_zip(zip_path, extract_to):
    if os.path.exists(zip_path):
        print(f"🗜️ {os.path.basename(zip_path)} 압축 해제 중...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"완료: {extract_to}")
    else:
        print(f"에러: {zip_path} 파일이 /content에 없습니다. 파일명을 확인해 주세요.")

# 1. Helmet, Head 데이터셋 압축 풀기
extract_zip('/content/project_v5.zip', '/content/project_v5')

# 2. Person 세그멘테이션 데이터셋 압축 풀기
extract_zip('/content/person_segment.zip', '/content/person_segment')

🗜️ project_v5.zip 압축 해제 중...
완료: /content/project_v5
🗜️ person_segment.zip 압축 해제 중...
완료: /content/person_segment


In [ ]:
import os
import json
import glob
import shutil
import random
import cv2

# 1. 경로 설정
v5_dataset_dir = '/content/project_v5/dataset'
person_dir = '/content/person_segment/person_segment'
output_dir = '/content/dataset_v6'

# 기존 찌꺼기 폴더 완벽 삭제
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# 2. YOLO 표준 구조 생성
splits = ['train', 'valid', 'test']
for split in splits:
    os.makedirs(os.path.join(output_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, split, 'labels'), exist_ok=True)

# 3. 데이터 분할 비율 정의를 위해 모든 원본 파일 리스트업 준비
# 이번에는 딕셔너리 매핑 방식의 한계를 깨고, 독립 리스트 구조로 안전하게 처리합니다.
all_tasks = [] # (소스이미지경로, 소스라벨타입, 소스라벨내용/경로)

# --- [파트 A] project_v5 (Helmet, Head) 안전 수집 ---
v5_images_base = os.path.join(v5_dataset_dir, 'images')
v5_labels_base = os.path.join(v5_dataset_dir, 'labels')

for root, _, files in os.walk(v5_images_base):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_p = os.path.join(root, file)
            relative_img_path = os.path.relpath(img_p, v5_images_base)
            relative_dir = os.path.dirname(relative_img_path)
            name_without_ext, _ = os.path.splitext(file)

            if relative_dir:
                lbl_p = os.path.join(v5_labels_base, relative_dir, name_without_ext + '.txt')
            else:
                lbl_p = os.path.join(v5_labels_base, name_without_ext + '.txt')

            all_tasks.append({
                'src_img': img_p,
                'filename': file,
                'type': 'v5',
                'lbl_p': lbl_p if os.path.exists(lbl_p) else None,
                'coco_anns': None,
                'img_w': None, 'img_h': None
            })

# --- [파트 B] person_segment (Person) 로보플로우 통합 JSON 완벽 파싱 ---
for split in splits:
    split_folder_path = os.path.join(person_dir, split)
    coco_json_path = os.path.join(split_folder_path, '_annotations.coco.json')

    if os.path.exists(coco_json_path):
        print(f"📦 {split.upper()} 폴더 내에서 로보플로우 통합 COCO JSON을 찾아냈습니다!")
        with open(coco_json_path, 'r') as f:
            coco_data = json.load(f)

        id_to_img_info = {img['id']: img for img in coco_data['images']}
        img_id_to_anns = {}
        for ann in coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in img_id_to_anns:
                img_id_to_anns[img_id] = []
            img_id_to_anns[img_id].append(ann)

        for img_id, img_info in id_to_img_info.items():
            file_name = img_info['file_name']
            img_p = os.path.join(split_folder_path, file_name)

            if os.path.exists(img_p):
                all_tasks.append({
                    'src_img': img_p,
                    'filename': file_name,
                    'type': 'person',
                    'lbl_p': None,
                    'coco_anns': img_id_to_anns.get(img_id, []),
                    'img_w': img_info['width'],
                    'img_h': img_info['height']
                })

# --- [파트 C] 무작위 섞기 및 8:1:1 배분 후 저장 ---
random.seed(42)
random.shuffle(all_tasks)

num_total = len(all_tasks)
num_val = int(num_total * 0.1)
num_test = int(num_total * 0.1)

print(f"🔄 수집 완료! 총 {num_total}장의 데이터를 섞어서 배분하는 중...")

for idx, task in enumerate(all_tasks):
    if idx < num_val:
        target_split = 'valid'
    elif idx < (num_val + num_test):
        target_split = 'test'
    else:
        target_split = 'train'

    # 파일명 충돌 예방 조치: 모든 파일에 고유 인덱스를 붙여 유일한 파일명 생성
    name_without_ext, ext = os.path.splitext(task['filename'])
    unique_filename = f"{name_without_ext}_{idx}{ext}"
    unique_lblname = f"{name_without_ext}_{idx}.txt"

    dst_img = os.path.join(output_dir, target_split, 'images', unique_filename)
    dst_lbl = os.path.join(output_dir, target_split, 'labels', unique_lblname)

    # 이미지 복사
    shutil.copy(task['src_img'], dst_img)

    # 라벨 조합 및 생성
    combined_lines = []

    # 1. v5 타입 처리 (helmet:0, head:1)
    if task['type'] == 'v5' and task['lbl_p']:
        with open(task['lbl_p'], 'r') as f:
            combined_lines.extend(f.readlines())

    # 2. person 타입 처리 (person:2)
    elif task['type'] == 'person' and task['coco_anns']:
        img_w, img_h = task['img_w'], task['img_h']
        for ann in task['coco_anns']:
            if 'bbox' in ann:
                x, y, w, h = ann['bbox']
                x_center = (x + w / 2) / img_w
                y_center = (y + h / 2) / img_h
                yolo_w = w / img_w
                yolo_h = h / img_h
                combined_lines.append(f"2 {x_center:.6f} {y_center:.6f} {yolo_w:.6f} {yolo_h:.6f}\n")

    # 라벨 쓰기
    with open(dst_lbl, 'w') as f:
        f.writelines(combined_lines)

print("=" * 50)
print(f"🎉 통합 데이터셋 세팅 완료! 위치: {output_dir}")
print(f"📊 TRAIN : {len(os.listdir(os.path.join(output_dir, 'train/images')))}장")
print(f"📊 VALID : {len(os.listdir(os.path.join(output_dir, 'valid/images')))}장")
print(f"📊 TEST  : {len(os.listdir(os.path.join(output_dir, 'test/images')))}장")
print("=" * 50)

In [ ]:
import os
import glob

train_labels = glob.glob('/content/dataset_v6/train/labels/*.txt')
valid_labels = glob.glob('/content/dataset_v6/valid/labels/*.txt')
test_labels = glob.glob('/content/dataset_v6/test/labels/*.txt')
all_labels = train_labels + valid_labels + test_labels

class_counts = {0: 0, 1: 0, 2: 0}
total_boxes = 0

for label_path in all_labels:
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if parts:
                    try:
                        class_id = int(parts[0])
                        if class_id in class_counts:
                            class_counts[class_id] += 1
                            total_boxes += 1
                    except: pass

print("=" * 50)
print(f"📈 [라벨 전수조사 결과] 총 라벨 파일 수: {len(all_labels)}개")
print(f"📦 전체 Bounding Box 개수: {total_boxes}개")
print("-" * 50)
print(f"🪖 Class 0 (Helmet) 개수 : {class_counts[0]}개")
print(f"🧑 Class 1 (Head) 개수   : {class_counts[1]}개")
print(f"🚶 Class 2 (Person) 개수 : {class_counts[2]}개")
print("=" * 50)

📈 [라벨 전수조사 결과] 총 라벨 파일 수: 8648개
📦 전체 Bounding Box 개수: 33169개
--------------------------------------------------
🪖 Class 0 (Helmet) 개수 : 18966개
🧑 Class 1 (Head) 개수   : 5785개
🚶 Class 2 (Person) 개수 : 8418개


In [ ]:
import yaml

yaml_path = '/content/dataset_v6/data.yaml'

data_config = {
    'train': '/content/dataset_v6/train/images',
    'val': '/content/dataset_v6/valid/images',
    'test': '/content/dataset_v6/test/images',
    'nc': 3,
    'names': ['helmet', 'head', 'person']
}

with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, allow_unicode=True, default_flow_style=False)

print("✅ data.yaml 최종 생성 완료!")

✅ data.yaml 최종 생성 완료!


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.6 MB/s eta 0:00:00


In [ ]:
from ultralytics import RTDETR

# 1. 공식 RT-DETR-l 모델 가중치 로드
model = RTDETR("rtdetr-l.pt")

# 2. 3개 클래스 균형 성장을 위한 정밀 튜닝 파라미터로 학습 시작
results = model.train(
    # [1. 데이터 및 하드웨어 세팅]
    data='/content/dataset_v6/data.yaml',   # 대통합 성공한 v6 경로
    epochs=100,                             # 완벽한 수렴을 위한 100 에포크
    imgsz=640,                              # 표준 입력 크기
    batch=16,                               # T4 GPU VRAM 안전지대 (OOM 방지)
    device=0,
    workers=4,

    # [2. 전이 학습 및 최적화 설정]
    pretrained=True,                        # 사람 형태 시각 능력 보존을 위한 프리트레인 연동
    optimizer='AdamW',                      # Transformer 최적의 파트너
    lr0=0.00015,                            # 가중치 파괴 및 NaN 에러 방지 안전 학습률
    lrf=0.01,                               # 최종 감소 비율
    warmup_epochs=3.0,                      # 초기 가중치 충격 완화 웜업

    # [3. 과적합 방지 규제화]
    dropout=0.1,                            # 특정 패턴 의존 방지 유지
    weight_decay=0.0005,                    # 모델 복잡도 제어 가중치 감쇠

    # [4. ★ 클래스 불균형 및 오검출 방지를 위한 손실 가중치 커스텀]
    box=8.5,                                # 바운딩 박스 위치 정확도 가중치 (기본 7.5에서 상향)
    cls=2.5,                                # ★ 소수 클래스(Head)의 분류 정확도를 높이기 위해 클래스 가중치 강화 (기본 1.0)

    # [5. ★ 산업 현장 소형 객체 및 중첩 특화 데이터 증강 (Augmentation)]
    mosaic=1.0,                             # 4장 합성으로 원거리 작은 안전모/맨머리 검출력 극대화
    mixup=0.2,                              # ★ 작업자 간 겹침(중첩) 현상 대비를 위해 0.2로 상향
    scale=0.6,                              # ★ 다양한 거리감(스케일 변화) 학습을 위해 크기 배율 확대 (0.5~0.6 권장)
    hsv_h=0.015,                            # 다양한 안전모 색상 대응
    hsv_v=0.4,                              # 그늘, 야간, 역광 등 현장 조도 변화 무력화
    degrees=10.0,                           # 미세 회전으로 작업자의 다양한 꺾임 자세 대비
    translate=0.1,                          # 앵글 가장자리에 잘린 객체 구별력 확보
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=8.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_v6/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, i

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100        21G      1.046     0.7583     0.3214          8       1280: 100% ━━━━━━━━━━━━ 865/865 1.2s/it 17:12
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 2.2it/s 12.4s
                   all        864       3328      0.102      0.528      0.145     0.0681

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      21.6G     0.6897     0.9004     0.1704          8       1280: 100% ━━━━━━━━━━━━ 865/865 1.2s/it 16:50
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 2.6it/s 10.4s
                   all        864       3328      0.127      0.555      0.158     0.0873

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     30/100      21.6G     0.3772     0.4093    0.09166          8       1280: 100% ━━━━━━━━━━━━ 865/865 1.2s/it 16:46
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 2.6it/s 10.3s
                   all        864       3328      0.895      0.861      0.914      0.605

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     31/100      21.5G      0.382     0.4038     0.0869          8       1280: 5% ╸─────────── 44/865 2.4it/s 51.4s<5:47


KeyboardInterrupt: 

In [ ]:
! pip install yt_dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.9 MB/s eta 0:00:00


In [ ]:
import yt_dlp

ydl_opts = {
    # 업로드한 쿠키 파일 경로를 지정합니다.
    'cookiefile': 'youtube_cookies.txt',
    'format': 'best',
    # 필요한 다른 옵션들...
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download(['https://www.youtube.com/watch?v=Uyn9k9kKkT4'])

[youtube] Extracting URL: https://www.youtube.com/watch?v=Uyn9k9kKkT4
[youtube] Uyn9k9kKkT4: Downloading webpage
[youtube] Uyn9k9kKkT4: Downloading tv downgraded player API JSON


ERROR: [youtube] Uyn9k9kKkT4: Requested format is not available. Use --list-formats for a list of available formats


DownloadError: ERROR: [youtube] Uyn9k9kKkT4: Requested format is not available. Use --list-formats for a list of available formats

In [ ]:
import cv2
import yt_dlp
from ultralytics import YOLO
import os

# 1. 테스트하고 싶은 유튜브 URL 입력
youtube_url = "https://youtu.be/Uyn9k9kKkT4?si=-ahk3_4pygpczgj2"
video_save_path = "/content/downloaded_video.mp4"

# 2. yt-dlp로 유튜브 영상 다운로드
ydl_opts = {
    'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
    'outtmpl': video_save_path,
    'quiet': True
}

print("📥 유튜브 영상 다운로드 중...")
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([youtube_url])
print("✅ 다운로드 완료!")

# 3. 모델 로드 (학습된 가중치 경로 지정)
# 학습 결과가 저장된 경로를 명시합니다.
model_path = '/content/best.pt'
model = YOLO(model_path)

# 4. 영상 처리를 위한 OpenCV 설정
cap = cv2.VideoCapture(video_save_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

output_path = '/content/output_safety_test.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print("🚀 모델로 객체 탐지 및 영상 합성 시작...")
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # conf=0.40: 40% 확률 이상만 탐지
    results = model(frame, conf=0.40, verbose=False)

    # 예측 결과 그리기
    annotated_frame = results[0].plot()
    out.write(annotated_frame)

    frame_count += 1
    if frame_count % 50 == 0:
        print(f"🎬 {frame_count}번째 프레임 처리 중...")

# 메모리 해제
cap.release()
out.release()
print(f"🎉 모든 프레임 탐지 완료! 결과 파일 생성됨 ➡️ {output_path}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📥 유튜브 영상 다운로드 중...


ERROR: [youtube] Uyn9k9kKkT4: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


DownloadError: ERROR: [youtube] Uyn9k9kKkT4: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

In [ ]:
#멀리 있는건 잘 잡지만 박스 떨림이 많으면서 person이 아닌것도 잡아서 재도전
from ultralytics import RTDETR

# 1. 공식 RT-DETR-l 모델 가중치 로드
model = RTDETR("rtdetr-l.pt")

# 2. 박스 안정성 및 배경 노이즈 억제 튜닝 파라미터로 학습
results = model.train(
    # [1. 데이터 및 기본 세팅]
    data='/content/dataset_v6/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,

    # [2. 최적화 및 규제화]
    pretrained=True,
    optimizer='AdamW',
    lr0=0.00015,
    lrf=0.01,
    warmup_epochs=3.0,
    dropout=0.1,
    weight_decay=0.0005,

    # [3. ★ 클래스 불균형 해결 및 정확도 강화]
    box=7.5,                # ★ 박스 안정성을 위해 기존 8.5에서 기본값 수준으로 하향 조정
    cls=2.5,                # 클래스 분류 정확도 유지

    # [4. ★ 박스 떨림(Jittering) 방지 및 배경 노이즈 억제 데이터 증강]
    mosaic=1.0,             # 소형 객체 검출 유지
    mixup=0.15,             # ★ 과도한 믹스업은 경계면을 흐리게 하므로 0.15로 미세 하향
    scale=0.5,              # ★ 스케일 변화 범위를 좁혀, 모델이 객체 크기에 너무 민감하게 반응하지 않도록 조정

    # [5. 현장 조도 및 자세 대응]
    hsv_h=0.015,
    hsv_v=0.4,
    degrees=5.0,            # ★ 너무 잦은 회전은 박스 좌표 값을 불안정하게 하므로 5도로 축소
    translate=0.1,

    # [6. 추론 안정성을 위한 NMS/IoU 설정 보완]
    iou=0.5                 # ★ 학습 시 iou 임계치를 설정하여 겹치는 박스에 대한 모델의 판단력 강화
)

Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_v6/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.5, keras=False, kobj=1.0, line_width=None, lr0=0.00015, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=

Exception in thread Thread-40 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_

optimizer: AdamW(lr=0.00015, momentum=0.937) with parameter groups 143 weight(decay=0.0), 206 weight(decay=0.0005), 226 bias(decay=0.0)
: 0% ──────────── 0/865  2.5s

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       1/50        20G      1.386      4.877     0.6916          4       1280: 0% ──────────── 0/1730  5.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      20.5G     0.9955     0.7964     0.3171          4       1280: 100% ━━━━━━━━━━━━ 1730/1730 1.6it/s 18:06
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 2.6it/s 10.3s
                   all        864       3328      0.151      0.425      0.132     0.0504

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      21.4G     0.6001      1.077     0.2338          4       1280: 0% ──────────── 0/1730  0.7s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       4/50      21.7G     0.4826     0.8163     0.1286          4       1280: 100% ━━━━━━━━━━━━ 1730/1730 1.6it/s 17:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 2.6it/s 10.3s
                   all        864       3328      0.802       0.66      0.719      0.447

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       5/50      14.4G     0.4951     0.4583     0.1299          4       1280: 0% ──────────── 0/1730  0.8s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       5/50      21.5G     0.5578     0.5814     0.1596          4       1280: 1% ──────────── 15/1730 1.8it/s 9.8s<15:58


KeyboardInterrupt: 

In [ ]:
import zipfile
import os

def extract_zip(zip_path, extract_to):
    if os.path.exists(zip_path):
        print(f"🗜️ {os.path.basename(zip_path)} 압축 해제 중...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"완료: {extract_to}")
    else:
        print(f"에러: {zip_path} 파일이 /content에 없습니다. 파일명을 확인해 주세요.")

extract_zip('/content/dataset_v6.zip', '/content/dataset_v6')

🗜️ dataset_v6.zip 압축 해제 중...
완료: /content/dataset_v6


In [ ]:
import os
import shutil
from ultralytics import RTDETR

# 1. 환경 초기화: 기존 runs 폴더 및 캐시 삭제
# 학습을 깨끗하게 새로 시작하기 위한 조치입니다.
if os.path.exists('/content/runs'):
    shutil.rmtree('/content/runs')
    print("🧹 기존 runs 폴더 삭제 완료.")

# Ultralytics 데이터셋 캐시 삭제 (데이터 경로 오류 방지)
if os.path.exists('/content/datasets'):
    shutil.rmtree('/content/datasets')
    print("🧹 데이터셋 캐시 삭제 완료.")

# 2. 모델 로드 및 학습 재개
# last.pt를 통해 어제까지의 학습 가중치를 그대로 가져옵니다.
model = RTDETR('/content/last.pt')

# 3. 학습 시작
# data 인자는 폴더 구조와 완벽히 일치하는 경로를 사용하세요.
# (이중 폴더 확인: /content/dataset_v6/dataset_v6/data.yaml이 맞는지 한 번 더 확인!)
model.train(
    data='/content/dataset_v6/dataset_v6/data.yaml',
    epochs=50,
    resume=True  # last.pt의 상태를 이어받음
)

Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_v6/dataset_v6/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.5, keras=False, kobj=1.0, line_width=None, lr0=0.00015, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=/content/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       5/50      11.5G      0.493     0.5813     0.1412          4       1280: 100% ━━━━━━━━━━━━ 1730/1730 1.6it/s 17:34
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 108/108 10.1it/s 10.7s
                   all        864       3328      0.816      0.774      0.798      0.486

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       6/50        12G     0.3875     0.6333     0.1631          4       1280: 0% ──────────── 0/1730  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       6/50        12G     0.4684     0.5308     0.1296          4       1280: 100% ━━━━━━━━━━━━ 1730/1730 1.6it/s 17:34
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 108/108 10.0it/s 10.7s
                   all        864       3328      0.832      0.816      0.834      0.528

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       7/50        12G     0.4697      0.501    0.09318          4       1280: 0% ──────────── 0/1730  0.6s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      22/50      11.9G     0.3787     0.4167     0.0941          4       1280: 100% ━━━━━━━━━━━━ 1730/1730 1.6it/s 17:35
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 108/108 10.0it/s 10.8s
                   all        864       3328      0.872      0.893      0.915      0.586

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      23/50        12G      0.365     0.3811    0.09177          4       1280: 0% ──────────── 0/1730  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      23/50        12G     0.3296     0.3891    0.07549          4       1280: 0% ──────────── 8/1730 2.9it/s 5.0s<9:51


KeyboardInterrupt: 

In [ ]:
from ultralytics import RTDETR

model = RTDETR('/content/last.pt')

# 학습 재개
model.train(
    resume=True,
    data='/content/dataset_v6/dataset_v6/data.yaml'
)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_v6/dataset_v6/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.5, keras=False, kobj=1.0, line_width=None, lr0=0.00015, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=/content/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      23/50      11.2G     0.3698     0.4081     0.0929          4       1280: 100% ━━━━━━━━━━━━ 1730/1730 1.6it/s 17:33
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 108/108 8.3it/s 13.0s
                   all        864       3328      0.884      0.877      0.907      0.572

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      24/50      11.3G     0.2806     0.3542     0.1133          4       1280: 0% ──────────── 0/1730  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      24/50      11.3G     0.3677     0.4038    0.08892          4       1280: 100% ━━━━━━━━━━━━ 1730/1730 1.6it/s 17:29
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 108/108 10.1it/s 10.7s
                   all        864       3328      0.871      0.889      0.915      0.582

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      25/50      11.3G     0.3619     0.3985    0.05085          4       1280: 0% ──────────── 0/1730  0.6s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/50      11.2G     0.3522     0.3905    0.08501          4       1280: 100% ━━━━━━━━━━━━ 1730/1730 1.6it/s 17:31
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 108/108 10.0it/s 10.8s
                   all        864       3328      0.865      0.891      0.909      0.576

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/50      11.3G     0.3939     0.3679     0.1916          4       1280: 0% ──────────── 0/1730  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/50      11.3G     0.3863     0.3919    0.09573          4       1280: 3% ──────────── 59/1730 1.7it/s 36.4s<16:46


KeyboardInterrupt: 

In [ ]:
import cv2
from ultralytics import RTDETR

# 1. 모델 불러오기 (content에 있는 best.pt 경로 지정)
model = RTDETR('/content/best.pt')

# 2. 업로드한 영상 경로 지정
video_path = '/content/Take Time to Take Care (Vehicular Safety).mp4'

# 3. 추론 실행 및 결과 저장
# show=False로 설정하여 코랩 환경에서 에러 방지 (save=True로 결과 파일 생성)
results = model.predict(
    source=video_path,
    save=True,
    conf=0.5,
    imgsz=1024,
    stream=True
)

# 4. 추론 과정 실행
for result in results:
    # 각 프레임별 결과를 처리하며 진행 상황 확인
    pass

print("✅ 추론이 완료되었습니다!")
print("왼쪽 파일 탐색기에서 [runs/detect/predict] 폴더를 확인해 보세요.")


video 1/1 (frame 1/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 1024x1024 2 persons, 54.5ms
video 1/1 (frame 2/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 1024x1024 (no detections), 52.3ms
video 1/1 (frame 3/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 1024x1024 (no detections), 52.2ms
video 1/1 (frame 4/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 1024x1024 1 person, 52.5ms
video 1/1 (frame 5/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 1024x1024 1 person, 52.5ms
video 1/1 (frame 6/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 1024x1024 1 person, 52.3ms
video 1/1 (frame 7/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 1024x1024 1 person, 52.2ms
video 1/1 (frame 8/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 1024x1024 2 persons, 52.3ms
video 1/1 (frame 9/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 1024x1024 1 person, 52.1ms
video 1/1 (